# XP Phase 3 (Crafter) on Colab
Use a GPU runtime (Runtime > Change runtime type). Results go to Drive; every unit is resumable, so after a disconnect rerun the cell you were on (at most one in-flight job restarts from zero, up to about an hour at S near the cap).
Shell commands below use literal paths on purpose: `$VAR` expansion came out empty in Colab.
The clone cell uses branch `crafter-reopen`; if the branch is merged first, change `-b crafter-reopen` to `-b main`.
Order: setup, benchmark, smoke, calibrate (stop and read the output), then edit `budget.steps` and `budget.deep_set` in `configs/crafter.yaml` in the repo, push, rerun the clone cell, then run, status, report.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf xp-agents && git clone --depth 1 -b crafter-reopen https://github.com/knguyen0923/xp-agents xp-agents
%cd /content/xp-agents
!pip install -q -r requirements-crafter.txt -e .

In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
import os
print(os.cpu_count(), 'cpus')

## Benchmark
Compare with the spike: area 32 CPU 589 steps/s, MPS 781.

In [ ]:
# The printed `throughput ... steps/s` line is the benchmark: a 2-run, 4096-step plain-PPO run on the smoke config.
# The number includes start-up, as the spike's benchmark did.
%cd /content/xp-agents
!python -m xp.crafter.calibrate --config configs/crafter-smoke.yaml --out /content/drive/MyDrive/xp-results/crafter_smoke_calibration.json

## Smoke
End-to-end check of every path on the smoke config (tiny budget, provisional results allowed).

In [ ]:
%cd /content/xp-agents
!python -m xp.run --phase crafter --config configs/crafter-smoke.yaml --out /content/drive/MyDrive/xp-results-smoke
!python -m xp.report --phase crafter --config configs/crafter-smoke.yaml --out /content/drive/MyDrive/xp-results-smoke --figs /content/drive/MyDrive/xp-figs-crafter-smoke --allow-provisional

In [ ]:
# Calibration: up to 3 x 3M steps, about 3 hours at 781 steps/s. Resumable only by rerunning from zero, so keep the tab alive.
%cd /content/xp-agents
!python -m xp.crafter.calibrate --config configs/crafter.yaml --out /content/drive/MyDrive/xp-results/crafter_calibration.json --workers 1

**STOP HERE.** Read S, the G1/G2 and deep-set verdicts and the hours estimate printed above and tell the user. If it says PHASE STOPS, write the calibration up as the result; do not continue. Otherwise, only after the user approves S and the cost: edit `budget.steps` and `budget.deep_set` in `configs/crafter.yaml`, commit and push, then rerun the clone cell.

In [ ]:
# Full Phase 3 run (tune, freeze, evaluate). Resumable.
%cd /content/xp-agents
!python -m xp.run --phase crafter --config configs/crafter.yaml --out /content/drive/MyDrive/xp-results --workers 1

In [ ]:
# Progress: 36 tuning jobs (json) + 20 evaluation seed jobs (csv).
import glob
base = '/content/drive/MyDrive/xp-results/crafter'
tune = len(glob.glob(base + '/tuning/jobs/*.json'))
evals = len(glob.glob(base + '/*__seed-*.csv'))
print(tune + evals, 'done of 56')
print(tune, 'of 36 tuning jobs;', evals, 'of 20 evaluation seed jobs')

In [ ]:
%cd /content/xp-agents
!python -m xp.report --phase crafter --config configs/crafter.yaml --out /content/drive/MyDrive/xp-results --figs /content/drive/MyDrive/xp-figs-crafter
from IPython.display import Image
Image('/content/drive/MyDrive/xp-figs-crafter/degradation.png')